In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
import pickle
from pathlib import Path
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

# Ruta base del proyecto
BASE = r'C:\Users\c.villalobos-ortiz\Desktop\AIModel'
os.chdir(BASE)

# Configurar detector de manos
MODELO_PATH = 'hand_landmarker.task'
opciones = HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODELO_PATH),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)

# aqui activamos la red neuronal y la instancio a detector
detector = HandLandmarker.create_from_options(opciones)
print('✅ Todo listo')

✅ Todo listo


In [2]:
import os
os.environ['GLOG_minloglevel'] = '3'

def extraer_keypoints_normalizados(ruta_imagen):
    imagen = cv2.imread(str(ruta_imagen))
    if imagen is None:
        return None, None
    
    imagen_rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    mp_imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)
    resultado = detector.detect(mp_imagen)
    
    if not resultado.hand_landmarks:
        return None, None
    
    mano = resultado.handedness[0][0].category_name
    puntos = resultado.hand_landmarks[0]
    keypoints = []
    for punto in puntos:
        keypoints.extend([punto.x, punto.y, punto.z])
    
    if mano == "Right":
        for i in range(0, 63, 3):
            keypoints[i] = 1 - keypoints[i]
    
    muñeca_x = keypoints[0]
    muñeca_y = keypoints[1]
    muñeca_z = keypoints[2]
    
    for i in range(0, 63, 3):
        keypoints[i]   -= muñeca_x
        keypoints[i+1] -= muñeca_y
        keypoints[i+2] -= muñeca_z
    
    return keypoints, mano

print('✅ Función definida')

✅ Función definida


In [3]:
import csv

datos = []
imagenes_ok = 0
imagenes_fail = 0

columnas = [f'{eje}{i}' for i in range(21) for eje in ['x', 'y', 'z']] + ['letra']

with open('data/keypoints_v2.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(columnas)

rutas = list(Path('lsm_dataset/MSL-ABC').rglob('*.jpg'))
total = len(rutas)
print(f'Total de imágenes: {total}')
print('⏳ Procesando...')

for i, ruta_imagen in enumerate(rutas):
    etiqueta = ruta_imagen.parent.name
    if len(etiqueta) != 1 or not etiqueta.isalpha():
        continue

    keypoints, mano = extraer_keypoints_normalizados(ruta_imagen)
    if keypoints is not None:
        datos.append(keypoints + [etiqueta])
        imagenes_ok += 1
    else:
        imagenes_fail += 1

    if imagenes_ok % 5000 == 0 and imagenes_ok > 0:
        with open('data/keypoints_v2.csv', 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerows(datos)
        datos = []
        print(f'💾 {imagenes_ok}/{total} ({imagenes_ok/total*100:.1f}%)')

if datos:
    with open('data/keypoints_v2.csv', 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(datos)

print(f'\n✅ Detectadas: {imagenes_ok}')
print(f'❌ Sin detección: {imagenes_fail}')
print(f'💾 keypoints_v2.csv guardado')

Total de imágenes: 279716
⏳ Procesando...
💾 5000/279716 (1.8%)
💾 10000/279716 (3.6%)
💾 15000/279716 (5.4%)
💾 20000/279716 (7.2%)
💾 25000/279716 (8.9%)
💾 30000/279716 (10.7%)
💾 35000/279716 (12.5%)
💾 40000/279716 (14.3%)
💾 45000/279716 (16.1%)
💾 50000/279716 (17.9%)
💾 55000/279716 (19.7%)
💾 60000/279716 (21.5%)
💾 65000/279716 (23.2%)
💾 70000/279716 (25.0%)
💾 75000/279716 (26.8%)
💾 80000/279716 (28.6%)
💾 85000/279716 (30.4%)
💾 90000/279716 (32.2%)
💾 95000/279716 (34.0%)
💾 100000/279716 (35.8%)
💾 105000/279716 (37.5%)
💾 110000/279716 (39.3%)
💾 115000/279716 (41.1%)
💾 120000/279716 (42.9%)
💾 125000/279716 (44.7%)
💾 130000/279716 (46.5%)
💾 135000/279716 (48.3%)
💾 140000/279716 (50.1%)
💾 145000/279716 (51.8%)
💾 150000/279716 (53.6%)
💾 155000/279716 (55.4%)
💾 160000/279716 (57.2%)
💾 165000/279716 (59.0%)
💾 170000/279716 (60.8%)
💾 175000/279716 (62.6%)
💾 180000/279716 (64.4%)
💾 185000/279716 (66.1%)
💾 190000/279716 (67.9%)
💾 195000/279716 (69.7%)
💾 200000/279716 (71.5%)
💾 205000/279716 (73.3%)

In [4]:
tamaño = os.path.getsize('data/keypoints_v2.csv') / (1024*1024)
df_v2 = pd.read_csv('data/keypoints_v2.csv')
print(f'📁 keypoints_v2.csv: {tamaño:.1f} MB')
print(f'📊 Filas: {len(df_v2)}')
print(f'📋 Columnas: {len(df_v2.columns)}')
print(df_v2['letra'].value_counts().sort_index())

📁 keypoints_v2.csv: 329.5 MB
📊 Filas: 276113
📋 Columnas: 64
letra
A    13818
B    13665
C    12916
D    13458
E    13429
F    13565
G    12971
H    12279
I    13479
L    13424
M    13001
N    12203
O    13057
P    13033
R    13296
S    12993
T    13500
U    13062
V    13019
W    13090
Y    12855
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df_v2.drop('letra', axis=1).values
y = df_v2['letra'].values

encoder_v2 = LabelEncoder()
y_encoded = encoder_v2.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'✅ Datos listos')
print(f'   Train: {X_train.shape}')
print(f'   Test:  {X_test.shape}')
print(f'   Clases: {encoder_v2.classes_}')

✅ Datos listos
   Train: (220890, 63)
   Test:  (55223, 63)
   Clases: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'L' 'M' 'N' 'O' 'P' 'R' 'S' 'T' 'U'
 'V' 'W' 'Y']


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

modelo_v2 = Sequential([
    Input(shape=(63,)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(21, activation='softmax')
])

modelo_v2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('models/v2/modelo_lsm_v2.keras', save_best_only=True, verbose=1)
]

os.makedirs('models/v2', exist_ok=True)

historial = modelo_v2.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

print("✅ Entrenamiento terminado")

Epoch 1/50
6194/6213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7870 - loss: 0.6581
Epoch 1: val_loss improved from None to 0.17656, saving model to models/v2/modelo_lsm_v2.keras
6213/6213 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.8571 - loss: 0.4311 - val_accuracy: 0.9373 - val_loss: 0.1766
Epoch 2/50
6210/6213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9038 - loss: 0.2853
Epoch 2: val_loss improved from 0.17656 to 0.15251, saving model to models/v2/modelo_lsm_v2.keras
6213/6213 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9067 - loss: 0.2759 - val_accuracy: 0.9476 - val_loss: 0.1525
Epoch 3/50
6207/6213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9140 - loss: 0.2522
Epoch 3: val_loss improved from 0.15251 to 0.13315, saving model to models/v2/modelo_lsm_v2.keras
6213/6213 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9162 - loss: 0.2465 - val_accuracy: 0.9549 - val_loss: 0.1332
Epoch 4/50
6202/6213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9208 - loss: 0.23